# Assignment: linear SVM — outlier sensitivity and high-dimensional geometry

В этом notebook-е оставлены две исследовательские задачи:

1. **Один выброс и зависимость решения от $C$**.
2. **Геометрия линейного SVM в больших размерностях**.

Идея комплекта: посмотреть, как линейный SVM реагирует на трудные точки,
и как меняются геометрические свойства обучающей выборки при росте размерности.


## Что сдавать

1. Заполненный notebook.
2. Короткие выводы после каждой задачи.
3. Для задачи 2 — аккуратный аналитический вывод байесовского классификатора и байесовской ошибки.
4. Для задачи 2 — краткое обсуждение того, как меняются разделимость обучающей выборки, число опорных векторов и качество при росте размерности.


## 0. Импорт и вспомогательные функции


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.datasets import make_blobs
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True


In [ ]:
def plot_points_2d(X, y, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(X[y == -1, 0], X[y == -1, 1], s=55, label="class -1", alpha=0.85)
    ax.scatter(X[y ==  1, 0], X[y ==  1, 1], s=55, label="class +1", alpha=0.85)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    if title is not None:
        ax.set_title(title)
    return ax

def plot_separator_and_margins(model, ax, xlim, color="tab:red", label=None):
    w = model.coef_[0]
    b = model.intercept_[0]
    xs = np.linspace(xlim[0], xlim[1], 500)

    if abs(w[1]) < 1e-12:
        x0 = -b / w[0]
        ax.axvline(x0, color=color, linewidth=2.5, label=label)
        return

    ys0 = -(w[0] * xs + b) / w[1]
    ys1 = -(w[0] * xs + b - 1.0) / w[1]
    ysm1 = -(w[0] * xs + b + 1.0) / w[1]

    ax.plot(xs, ys0, color=color, linewidth=2.5, label=label)
    ax.plot(xs, ys1, color=color, linewidth=1.2, linestyle="--", alpha=0.9)
    ax.plot(xs, ysm1, color=color, linewidth=1.2, linestyle="--", alpha=0.9)

def hinge_slacks(X, y, model):
    margins = y * model.decision_function(X)
    slacks = np.maximum(0.0, 1.0 - margins)
    return margins, slacks

def geometric_margin_from_model(model):
    w = model.coef_[0]
    return 1.0 / np.linalg.norm(w)

def make_outlier_data(random_state=42, n_bad=1):
    X_base, y01 = make_blobs(
        n_samples=[22, 22],
        centers=[(-2.0, -1.4), (2.1, 1.6)],
        cluster_std=[0.65, 0.65],
        random_state=random_state,
    )
    y = np.where(y01 == 0, -1, 1)

    X_bad = np.array([
        [0.9, 0.9],
        [1.2, 1.1],
        [1.45, 0.55],
    ])[:n_bad]
    y_bad = -np.ones(n_bad, dtype=int)

    X = np.vstack([X_base, X_bad])
    y = np.concatenate([y, y_bad])

    return X, y, X_bad

def generate_hd_sample(n, d, mu=0.7, sigma=1.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    y = rng.choice([-1, 1], size=n)
    X = sigma * rng.normal(size=(n, d))
    X[:, 0] += mu * y
    return X, y


## 1. Один выброс и зависимость решения от $C$

Рассмотрим двумерную выборку: два хорошо разделимых облака и одна точка с **противоречащей меткой**.

Требуется:

1. Обучить линейный SVM при нескольких значениях
   $$
   C \in \{10^{-2}, 10^{-1}, 1, 10, 10^2, 10^3\}.
   $$
2. Для каждого $C$ вычислить:
   - $\|w\|$;
   - ширину полосы $2/\|w\|$;
   - число support vectors;
   - средний hinge loss;
   - train accuracy.
3. Построить 3-4 характерных графика и объяснить, при каких $C$ модель:
   - фактически **игнорирует** выброс;
   - начинает **подстраиваться** под него.


In [ ]:
# Данные для задачи 1
X_out, y_out, X_bad = make_outlier_data(random_state=RANDOM_STATE, n_bad=1)

fig, ax = plt.subplots(figsize=(7, 5))
plot_points_2d(X_out, y_out, ax=ax, title="Данные с одним противоречащим выбросом")
ax.scatter(
    X_bad[:, 0], X_bad[:, 1],
    s=220, marker="*", edgecolors="black", linewidths=1.2,
    label="outlier"
)
ax.legend()
plt.show()


In [ ]:
# TODO:
# 1. Пройдите по сетке C_values.
# 2. Для каждого C обучите SVC(kernel="linear", C=C).
# 3. Соберите таблицу со столбцами:
#    C, ||w||, margin_width = 2/||w||, n_support_vectors, mean_hinge, train_accuracy.
# 4. Попробуйте найти диапазон C, где поведение модели резко меняется.

C_values = [1e-2, 1e-1, 1, 10, 1e2, 1e3]

results_outlier = pd.DataFrame(
    columns=[
        "C",
        "||w||",
        "margin_width",
        "n_support_vectors",
        "mean_hinge",
        "train_accuracy",
    ]
)

results_outlier


In [ ]:
# TODO:
# Выберите 3-4 характерных значения C и постройте для них графики:
# - точки двух классов;
# - разделяющая прямая;
# - margin-линии;
# - support vectors.
#
# Подсказка:
# model.support_vectors_ содержит опорные векторы.

selected_C = [1e-2, 1, 10, 1e3]

# Ваш код здесь


### Комментарий к задаче 1

Коротко ответьте письменно:

1. При каких $C$ выброс вносит минимальный вклад в решение?
2. При каких $C$ граница начинает заметно поворачиваться?
3. Почему увеличение $C$ часто приводит к **сужению** полосы margin?


## 2. Геометрия линейного SVM в больших размерностях

Рассмотрим вероятностную модель
$$
Y \in \{-1,1\}, \qquad \mathbb{P}(Y=1)=\mathbb{P}(Y=-1)=\frac12,
$$
$$
X = \mu Y e_1 + \sigma Z, \qquad Z \sim \mathcal{N}(0, I_d),
$$
где $e_1=(1,0,\dots,0)$.

Здесь **только первая координата** несёт полезный сигнал, а остальные $d-1$ координат — чистый шум.

Требуется:

1. Аналитически вывести байесовский классификатор.
2. Вычислить байесовскую ошибку.
3. Объяснить, почему эта ошибка **не зависит** от размерности $d$.
4. Зафиксировать
$$
n_{\mathrm{train}}=80,\qquad n_{\mathrm{test}}=5000,\qquad R=30.
$$
Для каждого
$$
d\in\{2,5,10,20,50,100\}
$$
30 раз независимо:
- сгенерировать обучающую и тестовую выборки из указанной модели;
- обучить линейный SVM на обучающей выборке;
- записать `train accuracy`, `test accuracy`, число support vectors и факт линейной разделимости обучающей выборки.

Затем для каждого $d$ усреднить результаты по 30 повторам и построить графики зависимости от $d$.

Основная цель — понять различие между:

- **истинной статистической сложностью** задачи;
- **геометрической лёгкостью** линейного разделения train-набора.


### Теоретическая часть

Подсказка: сравните условные плотности $p(x \mid Y=1)$ и $p(x \mid Y=-1)$.
Удобно взять логарифм отношения правдоподобий.

В этой модели
$$
X \mid (Y=1) \sim \mathcal{N}(\mu e_1, \sigma^2 I_d),
\qquad
X \mid (Y=-1) \sim \mathcal{N}(-\mu e_1, \sigma^2 I_d).
$$

Нужно показать, что байесовское правило зависит только от первой координаты.


In [ ]:
# Параметры эксперимента для задачи 2
mu = 0.7
sigma = 1.0
dims = [2, 5, 10, 20, 50, 100]
n_train = 80
n_test = 5000
n_repeats = 30
C_hd = 1.0

print("Проверьте аналитически, что Bayes error должна равняться Phi(-mu/sigma).")
print("Для каждого d эксперимент нужно повторить n_repeats раз независимо.")


In [ ]:
# TODO:
# Для каждого d и для каждого повтора:
#   1. Сгенерируйте train и test из модели
#        X = mu * Y * e1 + sigma * Z,  Z ~ N(0, I_d).
#   2. Обучите линейный SVM на train.
#   3. Вычислите:
#        - train_accuracy,
#        - test_accuracy,
#        - n_support_vectors.
#   4. Вычислите индикатор separable_train:
#        train-набор считается линейно разделимым, если линейный SVM
#        с очень большим C классифицирует все train-объекты без ошибок.
#
# Сохраните результаты в таблицу results_hd
# со столбцами, например:
#   d, repeat, train_accuracy, test_accuracy,
#   n_support_vectors, separable_train

results_hd = pd.DataFrame(
    columns=[
        "d",
        "repeat",
        "train_accuracy",
        "test_accuracy",
        "n_support_vectors",
        "separable_train",
    ]
)

# Ваш код здесь


In [ ]:
# TODO:
# Для каждого d усредните результаты по всем повторам и постройте графики:
#   1. средние train/test accuracy;
#   2. среднее число support vectors;
#   3. долю линейно разделимых train-выборок:
#         separable_train_rate(d) = average of separable_train over repeats.
#
# После этого кратко сопоставьте результаты
# с аналитическим выводом байесовского классификатора.

# Ваш код здесь


### Комментарий к задаче 2

Коротко обсудите:

1. Почему истинная сложность задачи в этой модели не зависит от $d$?
2. Почему при росте $d$ train-выборка может становиться легче линейно разделимой?
3. Почему это **не означает**, что test accuracy обязана монотонно расти?